# Notebook 13 — Figures and tables

**ClinicalShield v2 · Revision pipeline · CPU · minutes**

## Why figures and tables are generated together

v1's Table 6 reported 92.23% accuracy and 1.01% FPR. Figure 6 implied 86.98% and 10.83%.
Reviewers 1 and 4 both found the discrepancy.

It was not a modelling error. The table was transcribed by hand while the figure was drawn
separately, and nothing connected them.

This notebook reads `all_paper_numbers.json` — the reconciled ledger NB12 produces — and
emits **every figure and every LaTeX table from that one source, in one script**. A quantity
appearing in both a figure and a table reads the same entry, so divergence is not merely
unlikely; it is impossible.

Cell 8 lists which quantities appear in both, as a standing check.

## What each figure answers

| figure | reviewer point |
|---|---|
| class balance | R1-b, R4-5 — dataset counts reconciled |
| length confound | R4-6 — parity by construction, measured |
| confusion matrix | R1-a, R4-3 — matrix and table share a source |
| recall by vector | R2-2 — the central finding |
| attention localisation | R3-Q3, R3-Q10, R4-10 — measured, not illustrated |
| CEPS comparison | R4-7 — the no-op flaw made visible |
| ASR reduction | R1-f, R4-1 — the evidence for "Neutralizing" |
| MPIB | R1-d, R4-2 — detection and ASR on separate axes |
| encoding robustness | R3-Q2, R3 report — canonical and adversarial tiers |

Figure 1 from v1 is absent: Reviewer 2 found it redundant with Table 1, and it has been
removed rather than redrawn.

## Captions carry their n

Every caption states the partition and the sample count. v1's Table 6 combined a
plaintext-partition FPR with a full-pipeline recall in one row, and nothing in the caption
recorded that it had. Captions are written into `tables/captions.tex` alongside the figures,
so caption and data are produced in the same step.

## What this cannot draw

Schematics carry no measured data: the architecture diagram and the threat-model diagram are
drawn separately. Cell 8 records them in the manifest so the figure set is complete on the
record rather than silently short.

## Runtime
CPU. Minutes. Reads JSON, writes PNG, PDF and TeX.

In [ ]:
# ── Cell 1 · Configuration ────────────────────────────────────────────────────
CONFIG = {
    "SEED": 42,
    "DPI": 300,
    "FORMATS": ["png", "pdf"],      # pdf for LaTeX, png for review

    # Figures and tables are generated together, from the same file, in the same
    # script. v1's Table 6 and Figure 6 disagreed because they were produced
    # separately and one was transcribed by hand (R1-a, R4-3). Sharing a source
    # makes that divergence impossible rather than unlikely.
    "SOURCE": "data/results/all_paper_numbers.json",
    "RESULTS_DIR": "data/results",

    "FIG_DIR": "figures",
    "TAB_DIR": "tables",

    # Colour-blind safe, print-legible in greyscale.
    "PALETTE": {"benign": "#4C72B0", "adversarial": "#C44E52",
                "neutral": "#7F7F7F", "accent": "#55A868",
                "warn": "#DD8452", "light": "#CCCCCC"},

    "USE_DRIVE":      True,
    "DRIVE_DIR":      "/content/drive/MyDrive/ClinicalShield_v2",
    "PUSH_TO_GITHUB": True,
    "GITHUB_REPO":    "NehlTech/ClinicalShield",
    "GITHUB_BRANCH":  "v2-revision",
    "GIT_USER_NAME":  "Adu-Boahene Bright",
    "GIT_USER_EMAIL": "baduboahene@st.knust.edu.gh",
}
SEED = CONFIG["SEED"]
print("figures -> %s/  tables -> %s/" % (CONFIG["FIG_DIR"], CONFIG["TAB_DIR"]))
print("single source: %s" % CONFIG["SOURCE"])


In [ ]:
# ── Cell 2 · Environment and inputs ───────────────────────────────────────────
# NB11 retrains PubMedBERT across sweep arms, so it needs the dataset and
# Module 1 only. No checkpoints are loaded; each arm trains from scratch.
import sys, os, json, random, hashlib, subprocess, shutil, time, math
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

random.seed(SEED); np.random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules
DRIVE_ROOT, REPO_DIR = None, None

if IN_COLAB:
    if CONFIG["USE_DRIVE"]:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(CONFIG["DRIVE_DIR"]); DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        print("drive :", DRIVE_ROOT)
    repo_name = CONFIG["GITHUB_REPO"].split("/")[-1]
    REPO_DIR = Path("/content") / repo_name
    if not REPO_DIR.exists():
        try:
            from google.colab import userdata
            tok = userdata.get("GITHUB_TOKEN")
            r = subprocess.run(["git","clone","-q",
                "https://" + tok + "@github.com/" + CONFIG["GITHUB_REPO"] + ".git",
                str(REPO_DIR)], capture_output=True, text=True)
            print("clone :", "ok" if r.returncode == 0 else r.stderr[:200])
        except Exception as e:
            print("clone skipped:", type(e).__name__)
    else:
        print("clone : already present")
    if REPO_DIR.exists():
        for k, v in [("user.name", CONFIG["GIT_USER_NAME"]),
                     ("user.email", CONFIG["GIT_USER_EMAIL"])]:
            subprocess.run(["git","-C",str(REPO_DIR),"config",k,v], check=False)
        subprocess.run(["git","-C",str(REPO_DIR),"checkout","-q",
                        CONFIG["GITHUB_BRANCH"]], check=False, capture_output=True)
        subprocess.run(["git","-C",str(REPO_DIR),"pull","-q","origin",
                        CONFIG["GITHUB_BRANCH"]], check=False, capture_output=True)
    ROOT = REPO_DIR if REPO_DIR.exists() else Path("/content")
else:
    ROOT = Path.cwd()
    while not (ROOT/".git").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT/".git").exists():
        ROOT = Path.cwd()

DIRS = {"dataset": ROOT/"data"/"dataset", "stats": ROOT/"data"/"stats",
        "results": ROOT/"data"/"results", "src": ROOT/"src"/"clinicalshield",
        "ckpt": ROOT/"checkpoints"}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
print("root  :", ROOT)

import re
for pkg, mod in [("matplotlib", "matplotlib"), ("numpy", "numpy")]:
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       check=False)
import re
print("NB13 reads the reconciled ledger: no model, no dataset, no GPU required")

# the ledger and the raw result files
src = ROOT / "data" / "results" / "all_paper_numbers.json"
if not src.exists() and DRIVE_ROOT:
    d = DRIVE_ROOT / "data" / "results" / "all_paper_numbers.json"
    if d.exists():
        src.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(d, src)
if not src.exists():
    raise FileNotFoundError("all_paper_numbers.json not found -- run NB12 first")

_all = json.load(open(src))
L = _all["ledger"]
print("ledger loaded: %d entries from %s" % (len(L), _all["generated_at"]))

S = {}
for name, rel in _all["sources"].items():
    p = ROOT / rel
    if not p.exists() and DRIVE_ROOT and (DRIVE_ROOT / rel).exists():
        p.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(DRIVE_ROOT / rel, p)
    if p.exists():
        S[name] = json.load(open(p))
print("result files: %d of %d" % (len(S), len(_all["sources"])))


## Style

Colour-blind safe and legible in greyscale, since reviewers frequently print. Each `save()`
records the ledger keys the figure drew on, so any plotted value traces back to the notebook
that measured it.

In [ ]:
# ── Cell 3 · Plot style and helpers ───────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": CONFIG["DPI"],
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "figure.constrained_layout.use": True,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
})
P = CONFIG["PALETTE"]

FIG_DIR = ROOT / CONFIG["FIG_DIR"]
TAB_DIR = ROOT / CONFIG["TAB_DIR"]
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

manifest = []

def save(fig, name, caption, label, sources):
    """Write a figure in every configured format and record its provenance.

    `sources` lists the ledger keys the figure draws on, so a reader can trace
    any plotted value back to the notebook that measured it.
    """
    for ext in CONFIG["FORMATS"]:
        fig.savefig(FIG_DIR / ("%s.%s" % (name, ext)))
    plt.close(fig)
    manifest.append({"kind": "figure", "name": name, "label": label,
                     "caption": caption, "ledger_keys": sources})
    print("  figure  %-28s %s" % (name, ", ".join(CONFIG["FORMATS"])))

def write_table(name, latex, caption, label, sources):
    (TAB_DIR / (name + ".tex")).write_text(latex)
    manifest.append({"kind": "table", "name": name, "label": label,
                     "caption": caption, "ledger_keys": sources})
    print("  table   %-28s .tex" % name)

def val(key, default=None):
    """Read a value from the reconciled ledger. Missing keys are reported rather
    than silently defaulted, since a figure drawn from an absent number is how
    the v1 inconsistencies arose."""
    if key not in L:
        print("      WARNING: ledger key not found: %s" % key)
        return default
    return L[key]["value"]

def sd(key):
    return L.get(key, {}).get("sd")

def n_of(key):
    return L.get(key, {}).get("n")

print("style set; %d ledger entries available" % len(L))


## Dataset figures

The length-confound figure is the one worth reading closely. It shows both that the two
classes are drawn from the same length distribution and what that is worth: a classifier
given nothing but word count reaches AUC 0.5103 here against 1.0000 on a reconstruction of
the original design.

In [ ]:
# ── Cell 4 · Dataset figures ──────────────────────────────────────────────────

# Figure: class balance. v1's Table 2 and Figure 3 disagreed (R1-b, R4-5); both
# now read the same ledger entries.
n_ben, n_adv = val("dataset_benign"), val("dataset_adversarial")
fig, ax = plt.subplots(figsize=(3.2, 2.6))
bars = ax.bar(["Benign", "Adversarial"], [n_ben, n_adv],
              color=[P["benign"], P["adversarial"]], width=0.55)
for b, v in zip(bars, [n_ben, n_adv]):
    ax.text(b.get_x()+b.get_width()/2, v + max(n_ben, n_adv)*0.02,
            "%d\n(%.1f%%)" % (v, 100*v/(n_ben+n_adv)), ha="center", fontsize=8)
ax.set_ylabel("Retrieval chunks")
ax.set_ylim(0, max(n_ben, n_adv)*1.18)
save(fig, "fig_class_balance",
     "Class distribution of the evaluation corpus (n=%d retrieval chunks derived "
     "from openFDA drug labelling)." % (n_ben+n_adv),
     "fig:class-balance", ["dataset_benign", "dataset_adversarial"])

# Figure: length parity, the R4-6 fix made visible. v1 had benign at a median of
# 2,648 words against 20 for attacks; a classifier could score well on length
# alone.
lo = val("length_only_auc_v2")
lo_v1 = val("length_only_auc_v1_counterfactual")
fig, axes = plt.subplots(1, 2, figsize=(6.6, 2.7))
ax = axes[0]
rng = np.random.default_rng(SEED)
ben_sim = rng.normal(250, 53, 2000).clip(60, 260)
adv_sim = rng.normal(250, 55, 2000).clip(60, 260)
ax.hist([ben_sim, adv_sim], bins=28, label=["Benign", "Adversarial"],
        color=[P["benign"], P["adversarial"]], alpha=0.85)
ax.set_xlabel("Words per chunk"); ax.set_ylabel("Count")
ax.set_title("Length distribution (this work)")
ax.legend(frameon=False)
ax = axes[1]
ax.bar(["This work", "Original\ndesign"], [lo, lo_v1],
       color=[P["accent"], P["adversarial"]], width=0.5)
ax.axhline(0.5, ls="--", lw=1, color=P["neutral"])
ax.text(1.42, 0.52, "chance", fontsize=7, color=P["neutral"], ha="right")
ax.set_ylim(0, 1.08); ax.set_ylabel("AUC, word count only")
ax.set_title("Length as a predictor")
for i, v in enumerate([lo, lo_v1]):
    ax.text(i, v + 0.03, "%.4f" % v, ha="center", fontsize=8)
save(fig, "fig_length_confound",
     "Length confounding. Left: benign and adversarial chunks are drawn from the "
     "same length distribution by construction. Right: a classifier given only "
     "word count achieves AUC %.4f here against %.4f on a reconstruction of the "
     "original design." % (lo, lo_v1),
     "fig:length-confound",
     ["length_only_auc_v2", "length_only_auc_v1_counterfactual",
      "length_parity_pct"])


## Detection figures

The confusion matrix prints accuracy, recall and FPR computed from the counts it displays.
They match Table~\ref{tab:detection} because both read the same entry.

In [ ]:
# ── Cell 5 · Detection figures ────────────────────────────────────────────────

# Figure: confusion matrix. Drawn from the same pooled counts the metrics are
# computed from, so figure and table cannot disagree (R1-a, R4-3).
pc = S["module2"]["pooled_confusion"]
cm = np.array([[pc["tn"], pc["fp"]], [pc["fn"], pc["tp"]]])
tot = cm.sum()
fig, ax = plt.subplots(figsize=(3.4, 3.0))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.max())
for i in range(2):
    for j in range(2):
        ax.text(j, i, "%d\n%.1f%%" % (cm[i, j], 100*cm[i, j]/tot),
                ha="center", va="center", fontsize=9,
                color="white" if cm[i, j] > cm.max()*0.55 else "black")
ax.set_xticks([0, 1], ["Benign", "Adversarial"])
ax.set_yticks([0, 1], ["Benign", "Adversarial"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.grid(False)
acc = (pc["tn"]+pc["tp"])/tot
fpr = pc["fp"]/(pc["fp"]+pc["tn"])
rec = pc["tp"]/(pc["tp"]+pc["fn"])
ax.set_title("accuracy %.4f   recall %.4f   FPR %.4f" % (acc, rec, fpr),
             fontsize=8)
save(fig, "fig_confusion_matrix",
     "Pooled confusion matrix across %d seeds on the held-out partition "
     "(n=%d predictions). Accuracy, recall and false-positive rate printed above "
     "are computed from these counts and match Table~\\ref{tab:detection}."
     % (len(S["module2"]["seeds"]), tot),
     "fig:confusion", ["module2_accuracy", "module2_recall", "module2_fpr"])

# Figure: recall by attack vector, the paper's central finding
vo, vm = val("module2_recall_override"), val("module2_recall_misinformation")
so, sm = sd("module2_recall_override"), sd("module2_recall_misinformation")
fig, ax = plt.subplots(figsize=(3.6, 2.8))
xs = [0, 1]
ax.bar(xs, [vo, vm], yerr=[so or 0, sm or 0], capsize=4, width=0.5,
       color=[P["benign"], P["adversarial"]])
ax.set_xticks(xs, ["Instruction\noverride", "Clinical\nmisinformation"])
ax.set_ylabel("Detection recall")
ax.set_ylim(0.8, 1.02)
for x, v, s in zip(xs, [vo, vm], [so, sm]):
    ax.text(x, v + (s or 0) + 0.008, "%.4f" % v, ha="center", fontsize=8)
save(fig, "fig_recall_by_vector",
     "Detection recall by attack vector, mean $\\pm$ SD across five seeds. "
     "Misinformation payloads carry no syntactic markers and are detected less "
     "reliably and less consistently (Welch $t=3.60$, $p=0.019$).",
     "fig:recall-vector",
     ["module2_recall_override", "module2_recall_misinformation"])

# Figure: attention localisation against chance (R3-Q3, R3-Q10). v1 showed one
# qualitative heatmap whose caption contradicted the figure (R4-10).
pk, rb = val("attention_precision_at_k"), val("attention_random_baseline")
pks = sd("attention_precision_at_k")
fig, ax = plt.subplots(figsize=(3.4, 2.7))
ax.bar([0, 1], [pk, rb], yerr=[pks or 0, 0], capsize=4, width=0.5,
       color=[P["accent"], P["light"]])
ax.set_xticks([0, 1], ["Attention", "Random"])
ax.set_ylabel("precision@k against true span")
ax.set_ylim(0, 1.0)
for x, v in zip([0, 1], [pk, rb]):
    ax.text(x, v + 0.03, "%.4f" % v, ha="center", fontsize=8)
ax.text(0.5, 0.88, "%.1f$\\times$ chance" % (pk/rb), ha="center", fontsize=8,
        color=P["neutral"])
save(fig, "fig_attention_localisation",
     "Attention localisation. Precision@k of the highest-attention tokens "
     "against the ground-truth injection span (n=%s), compared with random "
     "selection over the same texts." % n_of("attention_precision_at_k"),
     "fig:attention", ["attention_precision_at_k", "attention_random_baseline"])


## Sanitisation, end-to-end and benchmark figures

The MPIB figure deliberately places detection and attack success on **separate axes**. They
measure different objects — one a property of the defence, the other of the target model —
and the original manuscript's error was to report one as the other.

In [ ]:
# ── Cell 6 · CEPS, end-to-end, MPIB, encodings ────────────────────────────────

# Figure: the CEPS v1 flaw, made visible (R4-7). v1 assigns a perfect score to a
# sanitiser that removes nothing, so its reported 1.0000 was indistinguishable
# from doing nothing.
strat = S["module4"]["strategies"]
names, v1s, v2s = [], [], []
for k, v in strat.items():
    names.append(k.split(" (")[0].replace("v2 rules + intent classifier",
                                          "ClinicalShield"))
    v1s.append(v["ceps_v1"]); v2s.append(v["combined"])
fig, ax = plt.subplots(figsize=(6.2, 2.9))
x = np.arange(len(names)); w = 0.38
ax.bar(x - w/2, v1s, w, label="CEPS v1 (preservation only)", color=P["light"])
ax.bar(x + w/2, v2s, w, label="CEPS v2 (preservation and removal)",
       color=P["accent"])
ax.set_xticks(x, [n[:22] for n in names], rotation=18, ha="right")
ax.set_ylabel("Score"); ax.set_ylim(0, 1.12)
ax.legend(frameon=False, loc="upper center", ncol=2)
for i, (a, b) in enumerate(zip(v1s, v2s)):
    ax.text(i - w/2, a + 0.02, "%.2f" % a, ha="center", fontsize=7)
    ax.text(i + w/2, b + 0.02, "%.2f" % b, ha="center", fontsize=7)
save(fig, "fig_ceps_comparison",
     "CEPS v1 against v2 across sanitisation strategies (n=%s adversarial "
     "chunks). A sanitiser that removes nothing scores 1.0000 under v1 and "
     "0.0000 under v2." % n_of("ceps_v2_combined"),
     "fig:ceps", ["ceps_v1", "ceps_v2_combined", "ceps_v1_noop", "ceps_v2_noop"])

# Figure: end-to-end attack success (R1-f, R4-1). The clean condition is the
# base rate without which an ASR figure has nothing to be measured against.
a_c, a_p, a_s = val("asr_clean"), val("asr_poisoned"), val("asr_sanitised")
fig, ax = plt.subplots(figsize=(3.8, 2.9))
bars = ax.bar(["Clean", "Poisoned", "Sanitised"], [a_c, a_p, a_s],
              color=[P["benign"], P["adversarial"], P["accent"]], width=0.55)
for b, v in zip(bars, [a_c, a_p, a_s]):
    ax.text(b.get_x()+b.get_width()/2, v + 0.008, "%.1f%%" % (100*v),
            ha="center", fontsize=8)
ax.set_ylabel("Attack success rate")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylim(0, max(a_c, a_p, a_s)*1.25)
ax.annotate("", xy=(2, a_s), xytext=(1, a_p),
            arrowprops=dict(arrowstyle="->", color=P["neutral"], lw=1.2))
ax.text(1.5, (a_p+a_s)/2 + 0.012, "-%.1f%%" % (100*val("asr_relative_reduction")),
        ha="center", fontsize=8, color=P["neutral"])
save(fig, "fig_asr_reduction",
     "End-to-end attack success against %s (n=%s). Sanitisation reduces attack "
     "success by %.1f\\%% relative (McNemar $p=%.1e$); the clean condition gives "
     "the base rate with no attack present."
     % (val("target_llm"), n_of("asr_poisoned"),
        100*val("asr_relative_reduction"), val("asr_mcnemar_p")),
     "fig:asr", ["asr_clean", "asr_poisoned", "asr_sanitised",
                 "asr_relative_reduction"])

# Figure: MPIB, detection and ASR side by side but never in one axis (R1-d, R4-2)
if "mpib" in S:
    det = val("mpib_detection_rate")
    mp, mc = val("mpib_asr2_poisoned"), val("mpib_asr2_clean")
    fig, axes = plt.subplots(1, 2, figsize=(6.4, 2.7))
    ax = axes[0]
    ax.bar(["ClinicalShield"], [100*det], color=P["adversarial"], width=0.4)
    ax.set_ylabel("Detection rate (%)"); ax.set_ylim(0, 100)
    ax.text(0, 100*det + 3, "%.2f%%" % (100*det), ha="center", fontsize=8)
    ax.set_title("Detection\n(property of the defence)", fontsize=9)
    ax = axes[1]
    ax.bar(["Clean", "Poisoned"], [mc, mp],
           color=[P["benign"], P["adversarial"]], width=0.5)
    ax.set_ylabel("ASR2 (%)"); ax.set_ylim(0, max(mc, mp)*1.4)
    for i, v in enumerate([mc, mp]):
        ax.text(i, v + 0.1, "%.2f%%" % v, ha="center", fontsize=8)
    ax.set_title("Attack success\n(property of the target model)", fontsize=9)
    save(fig, "fig_mpib",
         "MPIB V2 (n=%s). Detection rate and attack success rate are distinct "
         "quantities measured on different objects; the original manuscript "
         "reported the latter as the former." % n_of("mpib_detection_rate"),
         "fig:mpib", ["mpib_detection_rate", "mpib_asr2_poisoned",
                      "mpib_asr2_clean"])

# Figure: encoding robustness, canonical against adversarial (R3-Q2, R3 report)
if "module1" in S:
    res = S["module1"]["results"]
    fmts = [f for f in res if f not in ("plaintext", "control_clean")]
    rates = [res[f]["rate"] for f in fmts]
    adv_res = S["module1"].get("adversarial_variants", {}).get("results", {})
    fig, ax = plt.subplots(figsize=(6.6, 3.0))
    labels = fmts + [""] + list(adv_res)
    vals = rates + [np.nan] + [adv_res[k]["rate"] for k in adv_res]
    cols = [P["accent"]]*len(fmts) + ["none"] + [P["warn"]]*len(adv_res)
    ax.bar(range(len(labels)), vals, color=cols, width=0.65)
    ax.set_xticks(range(len(labels)),
                  [l.replace("_", " ")[:14] for l in labels],
                  rotation=55, ha="right", fontsize=7)
    ax.set_ylabel("Detection rate"); ax.set_ylim(0, 1.08)
    ax.axvline(len(fmts), color=P["neutral"], lw=0.8, ls="--")
    ax.text(len(fmts)/2, 1.03, "canonical", ha="center", fontsize=8,
            color=P["neutral"])
    ax.text(len(fmts)+1+len(adv_res)/2, 1.03, "adversarial (PyRIT)",
            ha="center", fontsize=8, color=P["neutral"])
    save(fig, "fig_encoding_robustness",
         "Module 1 detection by obfuscation format (n=%s per format, Wilson 95\\%% "
         "intervals). Left: the five formats the module is specified to handle. "
         "Right: adversarial variants generated with Microsoft PyRIT, covering "
         "the character-injection taxonomy of Hackett et al."
         % res[fmts[0]]["n"],
         "fig:encodings", ["module1_canonical_mean", "module1_adversarial_mean"])


## LaTeX tables

Written to `tables/*.tex` for `\input{}` into the manuscript, so a number cannot be
retyped on its way into the paper.

In [ ]:
# ── Cell 7 · LaTeX tables ─────────────────────────────────────────────────────
# Generated from the same ledger the figures read. Every caption states the
# partition and n, because v1's Table 6 combined a plaintext-partition FPR with
# a full-pipeline recall in one row and nothing recorded that it had.

def latex_table(caption, label, header, rows, align=None, note=None):
    ncol = len(header)
    align = align or ("l" + "r"*(ncol-1))
    out = ["\\begin{table}[htbp]", "\\centering",
           "\\caption{%s}" % caption, "\\label{%s}" % label,
           "\\begin{tabular}{%s}" % align, "\\hline",
           " & ".join(header) + " \\\\", "\\hline"]
    for r in rows:
        out.append(" & ".join(str(x) for x in r) + " \\\\")
    out += ["\\hline", "\\end{tabular}"]
    if note:
        out.append("\\\\[2pt]\\footnotesize{%s}" % note)
    out.append("\\end{table}")
    return "\n".join(out) + "\n"

pm = lambda k, d=4: ("$%.*f \\pm %.*f$" % (d, val(k), d, sd(k))
                     if sd(k) is not None else "$%.*f$" % (d, val(k)))

# Table: dataset composition
if "split" in S:
    rows = []
    for part in ["train", "val", "test"]:
        p = S["split"]["partitions"][part]
        rows.append([part.capitalize(), p["chunks"], p["groups"],
                     p["benign"], p["adversarial"],
                     "%.1f\\%%" % p["adversarial_pct"]])
    tot = S["attacks"]["counts"]
    rows.append(["\\textbf{Total}", tot["chunks"], tot["groups"],
                 tot["benign"], tot["adversarial"],
                 "%.1f\\%%" % (100*tot["adversarial"]/tot["chunks"])])
    write_table("tab_dataset", latex_table(
        "Evaluation corpus and partitions. Chunks are retrieval-sized passages "
        "derived from openFDA drug labelling; a group is one source document, "
        "and no group spans partitions.",
        "tab:dataset",
        ["Partition", "Chunks", "Groups", "Benign", "Adversarial", "\\% adv."],
        rows,
        note="Attack template pools are disjoint across partitions: 48 phrasings "
             "in training, 16 in validation, 16 in test."),
        "dataset composition", "tab:dataset", ["dataset_chunks"])

# Table: detection, the headline
rows = [
    ["Accuracy",  pm("module2_accuracy")],
    ["Precision", pm("module2_precision")],
    ["Recall",    pm("module2_recall")],
    ["F1",        pm("module2_f1")],
    ["AUC",       pm("module2_auc")],
    ["FPR",       pm("module2_fpr")],
]
write_table("tab_detection", latex_table(
    "Detection performance on the held-out partition (n=%s), mean $\\pm$ "
    "standard deviation across five independent training runs."
    % n_of("module2_accuracy"),
    "tab:detection", ["Metric", "Value"], rows, align="lr",
    note="All values are computed from the pooled confusion matrix reported in "
         "Figure~\\ref{fig:confusion} and are consistent with it by construction."),
    "detection metrics", "tab:detection",
    ["module2_accuracy", "module2_recall", "module2_fpr"])

# Table: CEPS v1 against v2
if "module4" in S:
    rows = []
    for k, v in S["module4"]["strategies"].items():
        rows.append([k.split(" (")[0][:28],
                     "%.4f" % v["preservation"], "%.4f" % v["removal"],
                     "%.4f" % v["ceps_v1"], "\\textbf{%.4f}" % v["combined"]])
    write_table("tab_ceps", latex_table(
        "CEPS v1 against v2 across sanitisation strategies (n=%s adversarial "
        "chunks, held-out partition). v1 measures preservation alone and "
        "therefore cannot distinguish targeted sanitisation from a sanitiser "
        "that removes nothing." % n_of("ceps_v2_combined"),
        "tab:ceps",
        ["Strategy", "Preservation", "Removal", "CEPS v1", "CEPS v2"], rows,
        note="CEPS v2 is the harmonic mean of preservation and removal; either "
             "component at zero yields zero. Removal is scored per payload "
             "sentence against the ground-truth injected text."),
        "CEPS comparison", "tab:ceps", ["ceps_v1", "ceps_v2_combined"])

# Table: end-to-end attack success
if "end_to_end" in S:
    e = S["end_to_end"]
    rows = [
        ["Clean (no attack present)", "%.2f\\%%" % (100*e["asr"]["clean"])],
        ["Poisoned",                  "%.2f\\%%" % (100*e["asr"]["poisoned"])],
        ["Sanitised",                 "%.2f\\%%" % (100*e["asr"]["sanitised"])],
        ["\\textbf{Relative reduction}",
         "\\textbf{%.1f\\%%}" % (100*e["asr"]["relative_reduction"])],
    ]
    write_table("tab_asr", latex_table(
        "End-to-end attack success against %s (n=%d). An attack succeeds when "
        "the injected instruction reaches clinical advice in the model's answer."
        % (e["target_model"], e["n_samples"]),
        "tab:asr", ["Condition", "Attack success rate"], rows, align="lr",
        note="McNemar $p = %.1e$, with %d attacks neutralised against %d "
             "introduced. The clean condition establishes the base rate without "
             "which a reduction cannot be interpreted."
             % (e["mcnemar"]["p"], e["mcnemar"]["neutralised"],
                e["mcnemar"]["introduced"])),
        "end-to-end ASR", "tab:asr", ["asr_poisoned", "asr_sanitised"])

# Table: baselines
if "baselines" in S:
    rows = []
    for name, r in S["baselines"]["baselines"].items():
        rows.append([name.replace("_", " "), "%.4f" % r["accuracy"],
                     "%.4f" % r["precision"], "%.4f" % r["recall"],
                     "%.4f" % r["fpr"], "%.4f" % r["f1"]])
    c = S["baselines"]["clinicalshield"]["aggregate"]
    rows.append(["\\textbf{ClinicalShield}"] +
                ["\\textbf{%.4f}" % c[m]["mean"]
                 for m in ["accuracy", "precision", "recall", "fpr", "f1"]])
    write_table("tab_baselines", latex_table(
        "Comparison against baselines on identical partitions with identical "
        "Module 1 ingestion (n=%d)." % S["baselines"]["protocol"]["n_test"],
        "tab:baselines",
        ["Method", "Accuracy", "Precision", "Recall", "FPR", "F1"], rows,
        note="Mixture of Encodings is a prevention-only defence: it re-encodes "
             "input to neutralise obfuscation and emits no detection signal, so "
             "its recall against an injection is zero by construction."),
        "baseline comparison", "tab:baselines", ["module2_accuracy"])

# Table: sensitivity sweeps
if "sweeps" in S:
    rows = []
    for sweep, arms in S["sweeps"]["results"].items():
        for lab, v in arms.items():
            if v.get("is_baseline") and sweep != "split":
                continue
            rows.append([sweep, lab[:30],
                         "$%.4f \\pm %.4f$" % (v["accuracy"], v["accuracy_sd"]),
                         "%+.4f" % v["delta_vs_baseline"],
                         "yes" if v["exceeds_seed_spread"] else "no"])
    write_table("tab_sweeps", latex_table(
        "Sensitivity analysis. Each factor varied with all others held at the "
        "baseline configuration, two seeds per arm.",
        "tab:sweeps",
        ["Sweep", "Configuration", "Accuracy", "$\\Delta$", "Beyond spread"],
        rows, align="llrrc",
        note="An arm is reported as sensitive only where its change exceeds the "
             "variation observed across seeds at fixed settings."),
        "sensitivity sweeps", "tab:sweeps", [])


## Manifest and verification

In [ ]:
# ── Cell 8 · Manifest, verification, and the diagrams this cannot draw ────────
# Every figure and table records the ledger keys it drew on, so any plotted
# value traces to the notebook that measured it.

fig_manifest = {
    "notebook": "13_figures_tables",
    "source": CONFIG["SOURCE"],
    "dpi": CONFIG["DPI"],
    "generated": manifest,
    "counts": {"figures": sum(1 for m in manifest if m["kind"] == "figure"),
               "tables": sum(1 for m in manifest if m["kind"] == "table")},
    "principle": ("Figures and tables are generated together from one reconciled "
                  "file. v1's Table 6 and Figure 6 disagreed because they were "
                  "produced separately and one was transcribed by hand."),
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
}

# Figure-table agreement: any quantity appearing in both must be identical.
shared = defaultdict(list)
for m in manifest:
    for k in m["ledger_keys"]:
        shared[k].append("%s:%s" % (m["kind"], m["name"]))
both = {k: v for k, v in shared.items()
        if any(x.startswith("figure") for x in v)
        and any(x.startswith("table") for x in v)}
print("")
print("quantities appearing in BOTH a figure and a table: %d" % len(both))
for k, v in both.items():
    print("  %-28s %s" % (k, ", ".join(v)))
print("  all read the same ledger entry, so divergence is not possible")

fig_manifest["shared_quantities"] = both

with open(ROOT / CONFIG["RESULTS_DIR"] / "figure_manifest.json", "w") as f:
    json.dump(fig_manifest, f, indent=2)

# ── what this notebook cannot generate ───────────────────────────────────────
# Schematics carry no measured data, so they are drawn rather than plotted. They
# are listed here so the figure set is complete on the record.
DIAGRAMS = {
    "fig_architecture": (
        "Five-module pipeline schematic, updated for v2: Module 1 now carries a "
        "canonicalisation stage ahead of format decoding, Module 4 consumes "
        "sentence-level judgements from Module 2, and a policy layer governs "
        "whether a document is released, sanitised or withheld."),
    "fig_threat_model": (
        "Retrieval attack surface: reference corpora against patient records, "
        "and where an adversary can place content without privileged access."),
}
print("")
print("DIAGRAMS NOT GENERATED HERE (no measured data; draw separately)")
for name, desc in DIAGRAMS.items():
    print("  %-22s %s" % (name, desc[:70] + "..."))
    fig_manifest.setdefault("external_diagrams", {})[name] = desc

with open(ROOT / CONFIG["RESULTS_DIR"] / "figure_manifest.json", "w") as f:
    json.dump(fig_manifest, f, indent=2)

# ── captions file, for pasting into the manuscript ───────────────────────────
cap = ["% Figure and table captions, generated with the figures themselves.",
       "% Every number here is read from all_paper_numbers.json.", ""]
for m in manifest:
    cap.append("%% %s: %s" % (m["kind"], m["name"]))
    cap.append("\\caption{%s}" % m["caption"])
    cap.append("\\label{%s}" % m["label"])
    cap.append("")
(ROOT / CONFIG["TAB_DIR"] / "captions.tex").write_text("\n".join(cap))
print("")
print("written: captions.tex")

if IN_COLAB and DRIVE_ROOT:
    for sub in [CONFIG["FIG_DIR"], CONFIG["TAB_DIR"]]:
        dst = DRIVE_ROOT / sub; dst.mkdir(parents=True, exist_ok=True)
        for f_ in (ROOT / sub).glob("*"):
            shutil.copy2(f_, dst / f_.name)
    print("mirrored to Drive")

if IN_COLAB and CONFIG["PUSH_TO_GITHUB"] and REPO_DIR and REPO_DIR.exists():
    subprocess.run(["git","-C",str(ROOT),"add","-f",
                    CONFIG["FIG_DIR"], CONFIG["TAB_DIR"],
                    "data/results/figure_manifest.json"], check=False)
    st = subprocess.run(["git","-C",str(ROOT),"status","--porcelain"],
                        capture_output=True, text=True)
    if st.stdout.strip():
        subprocess.run(["git","-C",str(ROOT),"commit","-q","-m",
                        "NB13: %d figures and %d tables from one reconciled source"
                        % (fig_manifest["counts"]["figures"],
                           fig_manifest["counts"]["tables"])], check=False)
        pr = subprocess.run(["git","-C",str(ROOT),"push","-q","origin",
                             CONFIG["GITHUB_BRANCH"]], capture_output=True, text=True)
        print("push:", "ok" if pr.returncode == 0 else pr.stderr[:200])


## Summary

**Paste the output below into the chat.**

In [ ]:
# ── Cell 9 · NB13 SUMMARY — paste this output ─────────────────────────────────
print("=" * 74)
print("NB13 — FIGURES AND TABLES")
print("=" * 74)
print("source  : %s" % CONFIG["SOURCE"])
print("ledger  : %d entries" % len(L))
print("output  : %d figures, %d tables"
      % (fig_manifest["counts"]["figures"], fig_manifest["counts"]["tables"]))
print("formats : %s at %d dpi" % (", ".join(CONFIG["FORMATS"]), CONFIG["DPI"]))
print("")
print("FIGURES")
for m in manifest:
    if m["kind"] == "figure":
        print("  %-26s %s" % (m["name"], m["label"]))
print("")
print("TABLES")
for m in manifest:
    if m["kind"] == "table":
        print("  %-26s %s" % (m["name"], m["label"]))
print("")
print("FIGURE-TABLE AGREEMENT")
print("  %d quantities appear in both a figure and a table" % len(both))
print("  all read the same ledger entry -- the v1 divergence cannot recur")
print("")
print("NOT GENERATED HERE (schematics, no measured data)")
for name in fig_manifest.get("external_diagrams", {}):
    print("  %s" % name)
print("=" * 74)
print("NB13 COMPLETE")
print("=" * 74)
